<a href="https://colab.research.google.com/github/hvnguyyen/ml-inventory-forecasting/blob/main/notebooks/03_baseline_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()

In [2]:
import pandas as pd
import numpy as np

calendar = pd.read_csv("calendar.csv")
sales = pd.read_csv("sales_train_validation.csv")
prices = pd.read_csv("sell_prices.csv")

print(calendar.shape)
print(sales.shape)
print(prices.shape)

(1969, 14)
(30490, 1919)
(6841121, 4)


In [3]:
sales_small = sales.head(100).copy()

id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [col for col in sales_small.columns if col.startswith("d_")]

print(sales_small.shape)
print(len(day_cols))
print(day_cols[:5], day_cols[-5:])

(100, 1919)
1913
['d_1', 'd_2', 'd_3', 'd_4', 'd_5'] ['d_1909', 'd_1910', 'd_1911', 'd_1912', 'd_1913']


In [4]:
series_0 = sales_small.iloc[0][day_cols].values.astype(np.float32)

print(series_0.shape)
print(series_0[:10])

(1913,)
[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [5]:
window_size = 30

X = []
y = []

for i in range(len(series_0) - window_size):
    X.append(series_0[i:i+window_size])
    y.append(series_0[i+window_size])

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

(1883, 30)
(1883,)


In [6]:
X[0], y[0]

(array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32),
 np.float32(0.0))

In [7]:
import torch
import torch.nn as nn

X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32)

print(X_tensor.shape)
print(y_tensor.shape)

torch.Size([1883, 30])
torch.Size([1883])


In [8]:
model = nn.Sequential(
    nn.Linear(window_size, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

print(model)

Sequential(
  (0): Linear(in_features=30, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=1, bias=True)
)


In [9]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [10]:
for epoch in range(10):
    model.train()

    predictions = model(X_tensor).squeeze()
    loss = criterion(predictions, y_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.7643
Epoch 2, Loss: 0.7164
Epoch 3, Loss: 0.6724
Epoch 4, Loss: 0.6321
Epoch 5, Loss: 0.5954
Epoch 6, Loss: 0.5623
Epoch 7, Loss: 0.5326
Epoch 8, Loss: 0.5062
Epoch 9, Loss: 0.4832
Epoch 10, Loss: 0.4633
